# 내 업무 Agent 직접 완성하기

이 노트북에서 조회 도구 → LangChain → Graph → 수정 루프 → MCP → A2A를 이어갑니다. 코드는 셀에 작성합니다. Python 파일로 옮기는 단계는 없습니다. 해당 장의 셀까지만 실행합니다. 미완성 셀의 NotImplementedError는 구현할 부분입니다.

함수를 고쳤다면 그 정의 셀을 먼저 실행하고 아래 연결·실행 셀도 다시 실행합니다. 저장은 Ctrl+S, 셀 실행은 Shift+Enter입니다. 키는 기존 workshop/.env에서 읽으며 셀에 입력하지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, json

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "build_lab" / "materials.py").is_file():
    raise RuntimeError("workshop/notebooks에서 이 노트북을 여십시오.")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from langchain.agents import create_agent
from langchain.tools import tool  # noqa: F401 - 아래 구현 셀에서 사용할 API
from course.policy_store import search_policy  # noqa: F401 - 아래 구현 셀에서 사용할 API
from course.common import POLICIES, get_model, trace_messages
from build_lab.materials import inspect_draft

print("실행 위치:", root)
print("정책 주제:", list(POLICIES))
from build_lab.notebook_checks import check_lookup, check_graph, check_accept_review


## 1A. 조회 도구 · 준비 운동

제공된 `search_policy(topic)`을 도구 함수에 연결합니다. CSV 읽기와 JSON 생성은 제공 함수가 맡습니다. 반환값을 직접 만든 JSON으로 대신하지 않습니다.

**작성:** `lookup_policy`에서 제공 함수를 호출하고 결과를 반환합니다. docstring에는 이 도구를 언제 쓰는지 적습니다.
**확인:** 아래 검사에서 정상·공백·미등록 입력의 정책 ID와 담당 팀까지 비교합니다. 이 단계는 연결 연습이며, 다음 단계에서 지침을 직접 설계합니다.


In [ ]:
def lookup_policy(topic: str) -> str:
    """Look up the current internal policy by topic, such as 정산 or 계정."""
    # 제공 search_policy(topic)을 호출하고 결과를 그대로 반환합니다.
    raise NotImplementedError("1A: 제공 search_policy를 연결하십시오.")

In [ ]:
check_lookup(lookup_policy)


## 1B. LangChain Agent

model, tools, system_prompt를 지정해 Agent를 반환합니다. 제공된 model과 policy_tool을 사용합니다. 도구 조회 시점과 정책이 없을 때의 행동을 지침에 적습니다. 다음 셀은 실제 모델 호출입니다.

In [ ]:
def build_agent(model, policy_tool):
    # model·tools·system_prompt를 지정합니다. 실제 호출은 호출자가 수행합니다.
    raise NotImplementedError("1B: create_agent로 도구를 가진 Agent를 구성하십시오.")

In [ ]:
model = get_model()
local_agent = build_agent(model, lookup_policy)
result = local_agent.invoke(
    {"messages": [{"role": "user", "content": '{"topic":"계정"}'}]},
    config={"recursion_limit": 12},
)
for entry in trace_messages(result["messages"]):
    print(json.dumps(entry, ensure_ascii=False, indent=2))

**완료 확인:** 도구 요청·도구 결과·최종 답변에서 P-02와 IT지원팀을 찾습니다. 질문을 없는업무로 바꿔 다시 실행해 담당 팀을 지어내지 않는지 확인합니다.

## 2. 업무 Graph · 분기와 추가 질문을 작성합니다

**요청:** 회신 대상이나 규정이 없으면 초안을 생성하지 않습니다. 다시 물을 때는 **부족한 정보만** 질문합니다. 계정 규정은 찾았는데 주소가 없으면 업무 주제를 다시 묻지 않아야 합니다.

먼저 아래 세 입력에서 무엇을 물어야 할지 예상합니다. 그 뒤 분기 함수와 질문 노드를 작성합니다.

|입력|추가로 물을 내용 · 실행 전에 작성|
|---|---|
|계정 / 공백 주소| |
|없는업무 / 정상 주소| |
|없는업무 / 공백 주소| |

`route_inquiry`는 다음 노드 이름 `draft` 또는 `ask`를 반환합니다. `ask_for_details`는 State에서 바꿀 필드를 반환합니다. 전체 노드 연결은 제공됩니다.

### 질문 노드의 입출력 계약

입력에서 `data['found']`와 `contact`를 읽습니다. 반환 dict에는 다음 필드가 필요합니다.

- `missing`: 부족한 필드 이름의 목록. 정책이 없으면 `topic`, 공백 연락처이면 `contact`; 둘 다이면 이 순서로 넣습니다.
- `draft`: `missing`에 해당하는 내용만 묻는 문장. 문장은 직접 정합니다.
- `decision`: `ask`, `history`: 빈 목록, `visited`: 기존 목록에 `ask`를 추가한 새 목록.

조회 결과와 기존 입력을 다시 만들거나 지우지 않습니다. 분기 함수와 노드의 반환값이 왜 다른지 설명할 수 있어야 합니다.


In [ ]:
def route_inquiry(state):
    """정책과 회신 대상을 읽고 draft 또는 ask로 분기합니다."""
    # 정책이 있어도 회신 대상이 없으면 생성하면 안 됩니다.
    raise NotImplementedError(
        "2: 정책 유무와 공백 연락처를 검사하는 분기를 구현하십시오."
    )


def ask_for_details(state):
    """부족한 정보만 질문하고 변경할 State 필드를 반환합니다."""
    raise NotImplementedError("2: missing과 질문, 방문 기록을 반환하십시오.")


def build_workflow(lookup, generate, refine, limit=2):
    """제공 그래프에 학생 분기를 연결합니다. guided.py의 노드·간선을 읽습니다."""
    from build_lab.guided import build_workflow as assemble

    return assemble(lookup, generate, refine, limit, router=route_inquiry, ask_node=ask_for_details)

In [ ]:
def generate_answer(topic):
    reply = local_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": json.dumps({"topic": topic}, ensure_ascii=False),
                }
            ]
        },
        config={"recursion_limit": 12},
    )
    return reply["messages"][-1].content


# 수정 루프를 배우기 전에는 실제 초안을 한 번 검토합니다.
def inspect_once(draft, data, limit):
    feedback = inspect_draft(draft, data)
    return {
        "status": "held" if feedback else "passed",
        "draft": draft,
        "history": [{"attempt": 0, "draft": draft, "feedback": feedback}],
    }


draft_calls = []


def generate(topic):
    draft_calls.append(topic)
    return generate_answer(topic)


graph = build_workflow(lookup_policy, generate, inspect_once)

### 2A. 내 구현을 검사하고 실제 Agent로 확인합니다

먼저 모델 없는 검사가 다섯 입력의 경로·부족한 필드·초안 생성 여부를 비교합니다. 이 검사는 생성 함수만 고정하며 **자신이 작성한 분기와 노드**를 실행합니다.

그다음 `topic`, `contact`를 바꿔 실제 Agent로 확인합니다. `초안 생성 호출`은 Agent를 실행한 횟수입니다. 도구 왕복 중의 모델 API 호출 횟수와는 다릅니다.

**완료:** 질문 문장을 읽고 이미 제공한 정보를 다시 묻지 않는지 확인합니다. 표에 없는 반례도 하나 골라 아래 셀에서 실행하고, 예상과 실제가 다른 첫 지점을 찾습니다.


In [ ]:
check_graph(build_workflow, lookup_policy)

topic = "계정"
contact = "   "
draft_calls.clear()
result = graph.invoke({"topic": topic, "contact": contact})
print("방문:", result["visited"])
print("판정:", result["decision"])
print("초안 생성 호출:", len(draft_calls))
print("답변:", result["draft"])
print("부족한 필드:", result.get("missing", []))


## 3. 제공 수정 Loop 관찰

실패한 초안을 실제 모델에 다시 맡깁니다. `refine_answer`는 제공 루프이며, 공통 과정에서는 그대로 사용합니다. 전체 알고리즘 작성은 선택 심화입니다.

**실행 전 예상:** 상한이 0이면 수정은 몇 번 실행될까요? 모델이 직전과 똑같은 초안을 반환하면 언제 멈출까요?

`passed`는 검토 통과, `stalled`는 직전과 동일한 초안, `held`는 수정 상한 소진입니다. 같은 오류가 남아 있어도 문자열이 달라지면 이 구현은 정체로 판정하지 않습니다. 다음 셀의 `limit`을 0과 2로 바꿔 `history`와 수정 입력을 비교합니다.


In [ ]:
def refine_answer(draft, data, revise, limit=2):
    """제공 루프를 사용합니다. 전체 알고리즘 작성은 선택 심화입니다."""
    from build_lab.guided import refine_answer as refine

    return refine(draft, data, revise, limit)

In [ ]:
feedback_inputs = []
data = json.loads(lookup_policy("계정"))


def make_reviser(policy_data, record):
    # 이번 문의의 정책을 사용하도록 수정 함수를 만듭니다.
    def revise(draft, feedback):
        record.append({"draft": draft, "feedback": feedback})
        return model.invoke(
            "규정에 따라 초안을 수정하십시오. 담당 팀과 근거 ID를 포함하십시오.\n"
            + json.dumps(
                {"policy": policy_data, "draft": draft, "feedback": feedback},
                ensure_ascii=False,
            )
        ).content

    return revise


revise = make_reviser(data, feedback_inputs)

limit = 2  # 0으로 바꾸면 수정 모델을 호출하지 않습니다.
repaired = refine_answer("확인 완료", data, revise, limit)
print(json.dumps(repaired, ensure_ascii=False, indent=2))
print("실제로 전달한 수정 입력:", feedback_inputs)
assert len(feedback_inputs) <= limit
assert bool(feedback_inputs) is (limit > 0)
assert repaired["history"][0]["draft"] == "확인 완료"

## 연결하고 설명합니다

앞에서 만든 그래프에 제공 수정 루프를 연결합니다. 결과가 통과하지 않으면 기준을 낮추지 말고 history의 실패와 수정 입력을 읽습니다.

In [ ]:
def refine(draft, data, limit):
    revise_current = make_reviser(data, [])
    return refine_answer(draft, data, revise_current, limit)


app = build_workflow(lookup_policy, generate, refine, limit=2)
final = app.invoke({"topic": "계정", "contact": "user@example.test"})
print(json.dumps(final, ensure_ascii=False, indent=2))

## Harness 설계 메모

교재 Harness의 [코딩 작업을 설계하는 활동](https://yo-sure.github.io/deepagents-handson/workshop/engineering#task)에서 공백 분기 결함과 출력 필드 안내 불일치의 두 사례를 읽습니다. 별도 활동지 파일 대신 **이 Markdown 셀에** 작성합니다. 더블클릭해 편집하고 Shift+Enter로 읽기 모드로 돌아옵니다.

**실행 순서 하나:** 문제 발견 → 수정 → 기능·문서 검토 → 결과 모으기 → 종료 순으로, 각 단계의 행동과 확인할 결과를 적습니다.

여기에 작성:

|반례|다음 행동|그 행동을 결정할 근거|
|---|---|---|
|같은 요청이 두 번 도착함| | |
|기능 또는 문서 검토 하나가 빠짐| | |
|수정하지 않은 초안을 PASS 처리함| | |

**마지막 확인:** 검토 둘이 같은 v2를 보는지, 누락된 검토를 성공으로 처리하지 않는지, 두 번 수정해도 실패하면 어디로 인계할지 확인합니다. 역할별 입력·산출물·권한의 상세 설계는 선택 확장입니다.


## 4. MCP · 원격 도구를 LangChain에 연결합니다

**할 일:** build_mcp_server에서 받은 조회 함수를 도구로 등록합니다. 다음 셀은 모델 없이 HTTP를 통해 원격 도구를 직접 호출합니다. 그다음 4B에서 동일한 도구를 Agent에 연결합니다.

서버 시작·종료는 제공된 serve_app이 맡습니다. 별도 터미널을 열지 않습니다. 함수를 수정하면 정의 셀부터 다시 실행합니다.


In [ ]:
from mcp.server.mcpserver import MCPServer  # noqa: F401 - 아래 구현 셀에서 사용할 API
from course.notebook_server import serve_app
from langchain.mcp import MCPAdapter
from fastmcp import Client


def build_mcp_server(policy_tool):
    raise NotImplementedError("MCPServer 생성 → policy_tool 등록 → 서버 반환")

In [ ]:
# 4A. 모델 없이 원격 도구 호출
server = build_mcp_server(lookup_policy)
async with serve_app(
    server.streamable_http_app(stateless_http=True, json_response=True)
) as url:
    async with MCPAdapter(Client(url + "/mcp", mode="2026-07-28")) as adapter:
        tools = await adapter.list_tools()
        print("도구 목록:", [t.name for t in tools])
        policy_tool = next(t for t in tools if t.name == "lookup_policy")
        print("입력 형식:", policy_tool.args)
        for topic in ["정산", "계정", "없는업무"]:
            content = await policy_tool.ainvoke({"topic": topic})
            print(topic, content)

**4A 완료 기준:** lookup_policy가 목록에 있고 topic 입력 형식이 보입니다. 정산은 P-01·재무지원팀, 계정은 P-02·IT지원팀, 없는업무는 found=false입니다. 도구의 반환값을 직접 출력한 것으로 모델 답변은 아닙니다.

## 4B. 같은 원격 도구를 Agent에 연결합니다

다음 셀의 tools 목록 조회와 create_agent의 tools 인자를 연결합니다. 질문을 바꿔 재실행합니다. 이 셀은 실제 모델을 호출하므로 앞에서 준비한 model과 API 연결이 필요합니다.


In [ ]:
question = "계정 문의는 어느 팀에 해야 하나요? 근거 ID도 알려주세요."
server = build_mcp_server(lookup_policy)
async with serve_app(
    server.streamable_http_app(stateless_http=True, json_response=True)
) as url:
    async with MCPAdapter(Client(url + "/mcp", mode="2026-07-28")) as adapter:
        tools = []  # TODO: adapter에서 도구 목록을 받아 연결합니다.
        remote_agent = create_agent(
            model=model,
            tools=tools,
            system_prompt="업무 규정을 도구로 조회하고 담당 팀과 정책 ID를 답합니다. 규정이 없으면 추가 확인을 요청합니다.",
        )
        result = await remote_agent.ainvoke(
            {"messages": [{"role": "user", "content": question}]}
        )
for message in result["messages"]:
    print(message.type, message.content)
    if getattr(message, "tool_calls", None):
        print("도구 요청:", message.tool_calls)

**4B 완료 기준:** AIMessage의 lookup_policy 요청 → ToolMessage의 P-02·IT지원팀 → 최종 답변을 확인합니다. 목록만 출력되거나 도구 요청 없이 답하면 아직 완료가 아닙니다. 없는 업무도 질문하여 추가 확인 안내를 비교합니다.


## 5. A2A · 발견 → 위임 → 결과 확인

5A는 모델 호출 없이 실제 HTTP로 Agent Card를 읽습니다. 5B에서는 같은 서버에 검토를 요청하며 실제 모델이 표현을 검토합니다. 서버 시작·종료와 SDK 연결 코드는 제공됩니다. 그 뒤 accept_review를 직접 구현합니다.


### 5A. Agent Card를 먼저 읽습니다 (모델 호출 없음)

이 셀은 서버가 공개한 기능과 접속 정보를 읽습니다. review-policy, JSONRPC, streaming=false를 찾습니다. model은 앞 장에서 준비한 접속 객체이며 Card 조회에서는 호출하지 않습니다.


In [ ]:
import httpx
from google.protobuf.json_format import MessageToDict
from a2a.client import A2ACardResolver
from course.a2a_lab import create_app, delegate
from course.notebook_server import serve_app

async with serve_app(lambda port: create_app(port, model=model), factory=True) as url:
    async with httpx.AsyncClient() as http:
        card = await A2ACardResolver(httpx_client=http, base_url=url).get_agent_card()
        print(json.dumps(MessageToDict(card), ensure_ascii=False, indent=2))

### 5B. 초안 하나를 실제로 맡깁니다 (모델 호출 있음)

draft에 담당 팀과 정책 ID가 있는 경우와 없는 경우를 비교합니다. Message와 Task는 SDK가 직렬화한 실제 값입니다. task.status.state는 프로토콜 표기, review.state는 수업 함수가 읽기 쉽게 바꾼 값입니다. 수업 서버는 최종 응답을 기다리므로 중간 상태 스트림은 출력하지 않습니다.


In [ ]:
import uuid

payload = {
    "topic": "계정",
    "draft": "계정 문의는 IT지원팀에 전달합니다. 근거: P-02",
    "request_id": str(uuid.uuid4()),
    "version": 1,
}
async with serve_app(lambda port: create_app(port, model=model), factory=True) as url:
    review = await delegate(url, payload)
print("보낸 Message:", json.dumps(review["message"], ensure_ascii=False, indent=2))
print("받은 Task:", json.dumps(review["task"], ensure_ascii=False, indent=2))
print("검토 산출물:", review["artifact"])

### 5C. 현재 초안에 쓸 수 있는 결과인지 판단합니다

**요청:** 검토 작업이 끝났더라도 다른 요청이나 옛 초안의 결과라면 채택하면 안 됩니다. `accept_review`를 직접 작성합니다.

|조건|반환값|
|---|---|
|submitted / working|pending|
|completed이며 현재 요청 ID·버전과 일치하고 passed가 불리언 True|accepted|
|그 외 상태, 결과 누락, 검토 실패, 요청·버전 불일치|held|

요청 ID는 공백이 아닌 문자열, 버전은 1 이상의 정수입니다. `True`를 버전 1로 받거나 문자열 `"true"`를 통과로 받지 않습니다.

**먼저 예상:** 옛 버전, 작업은 완료됐지만 검토 실패, 산출물 누락의 세 경우를 말로 판정합니다. 구현 후 아래 검사에서 이유별 결과를 확인합니다. 제공 사례 외에 잘못 수용될 수 있는 입력 하나를 직접 추가합니다.


In [ ]:
def accept_review(state, artifact, request_id, version):
    raise NotImplementedError("상태·요청 ID·버전·통과 여부를 확인하십시오.")

In [ ]:
check_accept_review(accept_review)

print(
    "실제 결과 수용:",
    accept_review(
        review["state"], review["artifact"], payload["request_id"], payload["version"]
    ),
)

## 6. 통합 · 한 요청을 세 단계로 확인합니다

앞에서 작성한 함수를 연결합니다. 입력 변경은 6A에서 하고 **6A → 6B → 6C를 순서대로** 실행합니다. 각 셀에서 나온 값을 다음 셀에 전달합니다.

### 6A. MCP에서 정책을 읽습니다

`topic`과 `contact`를 정합니다. 결과 `snapshot`의 업무명과 정책을 확인합니다. 아직 모델을 호출하지 않습니다.


In [ ]:
import uuid

topic = "계정"
contact = "user@example.test"
server = build_mcp_server(lookup_policy)
async with serve_app(
    server.streamable_http_app(stateless_http=True, json_response=True)
) as url:
    async with MCPAdapter(Client(url + "/mcp", mode="2026-07-28")) as adapter:
        remote_tools = await adapter.list_tools()
        remote_lookup = next(t for t in remote_tools if t.name == "lookup_policy")
        content = await remote_lookup.ainvoke({"topic": topic})
snapshot = json.loads(
    next(block["text"] for block in content if block["type"] == "text")
)

print("조회한 정책:", snapshot)


### 6B. 조회한 정책으로 Graph를 실행합니다

아래 연결 코드는 제공됩니다. 6A의 `snapshot`을 도구와 검토에 전달합니다. `visited`, `missing`, `history`, `decision`을 읽습니다. 정상 입력에서는 실제 모델을 호출합니다.


In [ ]:
def make_snapshot_tool(snapshot):
    def lookup_policy(topic: str) -> str:
        """이번 문의에서 조회한 정책을 반환합니다."""
        data = (
            snapshot
            if topic.strip() == snapshot["topic"]
            else {"found": False, "topic": topic.strip(), "policy": None}
        )
        return json.dumps(data, ensure_ascii=False)

    return lookup_policy


snapshot_lookup = make_snapshot_tool(snapshot)
connected_agent = build_agent(model, snapshot_lookup)


def connected_generate(topic):
    result = connected_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": json.dumps({"topic": topic}, ensure_ascii=False),
                }
            ]
        }
    )
    return result["messages"][-1].content


def connected_refine(draft, data, limit):
    def revise_current(draft, feedback):
        return model.invoke(
            "규정에 맞게 초안을 수정하십시오.\n"
            + json.dumps(
                {"policy": data, "draft": draft, "feedback": feedback},
                ensure_ascii=False,
            )
        ).content

    return refine_answer(draft, data, revise_current, limit)


workflow = build_workflow(snapshot_lookup, connected_generate, connected_refine)
result = workflow.invoke({"topic": snapshot["topic"], "contact": contact})
print("Graph 결과:", result)

### 6C. 통과한 초안만 원격 검토에 맡깁니다

6B 결과가 `passed`일 때 A2A 검토를 요청하고 자신의 `accept_review`로 수용 여부를 결정합니다. 추가 정보가 필요하거나 초안 검토가 실패했다면 원격 검토 전에 끝납니다.


In [ ]:
if result["decision"] == "passed":
    payload = {
        "topic": result["data"]["topic"],
        "draft": result["draft"],
        "request_id": str(uuid.uuid4()),
        "version": 1,
    }
    async with serve_app(
        lambda port: create_app(port, model=model), factory=True
    ) as url:
        review = await delegate(url, payload)
    decision = accept_review(
        review["state"], review["artifact"], payload["request_id"], payload["version"]
    )
    print("원격 검토:", review)
    print("수용 판단:", decision)
else:
    print("원격 검토 전 종료:", result["decision"])

**완료 확인:** 계정+정상 주소와 정보 부족 입력을 비교합니다. `completed`만으로 채택하지 않고 요청·버전·통과 조건을 확인합니다.

별칭 변경 과제에서는 `lookup_policy`를 수정한 뒤 정의 셀과 6A→6B→6C를 다시 실행합니다. 기존 정산·계정·미등록 입력의 동작도 유지되어야 합니다.
